In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.read_csv(r"C:\Users\Administrator\Downloads\differentiated+thyroid+cancer+recurrence\Thyroid_Diff.csv")
df.head()

,Age,Gender,Smoking,Hx Smoking,Hx Radiothreapy,Thyroid Function,Physical Examination,Adenopathy,Pathology,Focality,Risk,T,N,M,Stage,Response,Recurred
0,27,F,No,No,No,Euthyroid,Single nodular goiter-left,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Indeterminate,No
1,34,F,No,Yes,No,Euthyroid,Multinodular goiter,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
2,30,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
3,62,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
4,62,F,No,No,No,Euthyroid,Multinodular goiter,No,Micropapillary,Multi-Focal,Low,T1a,N0,M0,I,Excellent,No


## Encode Target ##

In [2]:
df['Recurred'] = df['Recurred'].map({'No': 0, 'Yes': 1})
df['Recurred'].value_counts()

Recurred
0    275
1    108
Name: count, dtype: int64

In [3]:
for col in ['Gender', 'Smoking', 'Hx Smoking', 'Hx Radiothreapy', 'Focality', 'M']:
    print(col, df[col].unique())

Gender ['F' 'M']
Smoking ['No' 'Yes']
Hx Smoking ['No' 'Yes']
Hx Radiothreapy ['No' 'Yes']
Focality ['Uni-Focal' 'Multi-Focal']
M ['M0' 'M1']


## Encode cột nhị phân ##

In [4]:
df['Gender'] = df['Gender'].map({'F': 0, 'M': 0})

for col in ['Smoking', 'Hx Smoking', 'Hx Radiothreapy']:
    df[col] = df[col].map({'No': 0, 'Yes': 1})

df['Focality'] = df['Focality'].map({'Uni-Focal': 0, 'Multi-Focal': 1})
df['M'] = df['M'].map({'M0': 0, 'M1': 1})

df[['Gender','Smoking','Hx Smoking','Hx Radiothreapy','Focality','M']].isnull().sum()

Gender             0
Smoking            0
Hx Smoking         0
Hx Radiothreapy    0
Focality           0
M                  0
dtype: int64

## Encode các cột thứ tự ##

In [5]:
for col in ['Risk', 'T', 'N', 'Stage']:
    print(col, sorted(df[col].unique()))

Risk ['High', 'Intermediate', 'Low']
T ['T1a', 'T1b', 'T2', 'T3a', 'T3b', 'T4a', 'T4b']
N ['N0', 'N1a', 'N1b']
Stage ['I', 'II', 'III', 'IVA', 'IVB']


In [6]:
risk_order = [['Low', 'Intermediate', 'High']]
t_order = [['T1a', 'T1b', 'T2', 'T3a', 'T3b', 'T4a', 'T4b']]
n_order = [['N0', 'N1a', 'N1b']]
stage_order = [['I', 'II', 'III', 'IVA', 'IVB']]

df['Risk'] = OrdinalEncoder(categories=risk_order).fit_transform(df[['Risk']])
df['T'] = OrdinalEncoder(categories=t_order).fit_transform(df[['T']])
df['N'] = OrdinalEncoder(categories=n_order).fit_transform(df[['N']])
df['Stage'] = OrdinalEncoder(categories=stage_order).fit_transform(df[['Stage']])

In [7]:
df[['Risk', 'T', 'N', 'Stage']].head(10)

,Risk,T,N,Stage
0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0


## Merge Adenopathy ##

In [10]:
df['Adenopathy'].value_counts()

Adenopathy
No           277
Right         48
Bilateral     32
Left          17
Extensive      7
Posterior      2
Name: count, dtype: int64

In [11]:
adenopathy_map = {
    'No': 'No',
    'Right': 'Unilateral',
    'Left': 'Unilateral',
    'Bilateral': 'Bilateral_Extensive',
    'Extensive': 'Bilateral_Extensive',
    'Posterior': 'Bilateral_Extensive'
}
df['Adenopathy'] = df['Adenopathy'].map(adenopathy_map)
df['Adenopathy'].value_counts()

Adenopathy
No                     277
Unilateral              65
Bilateral_Extensive     41
Name: count, dtype: int64

## One hot encode các cột nominal ##

In [12]:
nominal_cols = ['Thyroid Function', 'Physical Examination', 'Adenopathy', 'Pathology', 'Response']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=False)
df.shape

(383, 33)

In [14]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [15]:
df.info()
df.isnull().sum().sum()   # phải ra 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 33 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Age                                               383 non-null    int64  
 1   Gender                                            383 non-null    int64  
 2   Smoking                                           383 non-null    int64  
 3   Hx Smoking                                        383 non-null    int64  
 4   Hx Radiothreapy                                   383 non-null    int64  
 5   Focality                                          383 non-null    int64  
 6   Risk                                              383 non-null    float64
 7   T                                                 383 non-null    float64
 8   N                                                 383 non-null    float64
 9   M                    

np.int64(0)

In [23]:
df.to_csv(r'C:\Users\Administrator\thyroid-cancer-recurrence\data\processed\thyroid_processed.csv', index=False)